# Topic Discovery: Rigorous Workflow

目标：机器只提出候选 topic 结构；人工独立归并并验证；最终形成可报告、可审计、可复跑的主题体系。

**流程顺序（不可跳过）**
1. 冻结语料 → `corpus_freeze.json`
2. HDBSCAN 候选比较 → `hdbscan_candidate_summary.csv`
3. **Topic-number selection（C_V 扫描 10–100）** → `topic_number_cv_curve.*`
4. 选定最终主模型后，才生成 enhanced review / coder sheets（`topic_review_sheet_final_nr{K}.csv`）
5. 两名 coder 独立标注 → 一致性 → 仲裁 → 回填与敏感性

**注意**：`mcs=50` 仅是 HDBSCAN 候选基座，不是最终主模型；`hdbscan_mcs50_*` 原型表仅供流程测试。

可复现 CLI：`./.venv/bin/python -m phase1.run_topic_discovery --step all`



In [1]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "phase1").exists():
    ROOT = next(p for p in ROOT.parents if (p / "phase1").exists())

RUN_ID = "2026-05-27_topic_merged_bge_hdbscan_sensitivity"
RUN_DIR = ROOT / "output" / "experiments" / RUN_ID
SHARED_CORPUS_PATH = RUN_DIR / "shared_analyzable_corpus.csv"
CANDIDATE_MCS = [30, 50, 80, 100]
BASE_MCS = 50  # HDBSCAN base for reduce_topics scan; not the final main model

OUT_DIR = RUN_DIR / "topic_discovery_review"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load frozen snapshot + final model selection if available
FREEZE_PATH = OUT_DIR / "corpus_freeze.json"
SEL_PATH = OUT_DIR / "final_model_selection.json"
FINAL_NR = None
if SEL_PATH.is_file():
    FINAL_NR = json.loads(SEL_PATH.read_text(encoding="utf-8"))["selected_target_nr_topics"]

print("ROOT =", ROOT)
print("RUN_DIR =", RUN_DIR)
print("OUT_DIR =", OUT_DIR)
print("FREEZE =", FREEZE_PATH.exists())
print("FINAL_NR =", FINAL_NR)

ROOT = /Users/yilin/project/2604-robotic_failure_research
RUN_DIR = /Users/yilin/project/2604-robotic_failure_research/output/experiments/2026-05-27_topic_merged_bge_hdbscan_sensitivity
OUT_DIR = /Users/yilin/project/2604-robotic_failure_research/output/experiments/2026-05-27_topic_merged_bge_hdbscan_sensitivity/topic_discovery_review


## 0. 可复现 CLI（推荐）

完整流程由 `phase1/topic_discovery.py` 驱动：

```bash
./.venv/bin/python -m phase1.run_topic_discovery --step freeze
./.venv/bin/python -m phase1.run_topic_discovery --step select
./.venv/bin/python -m phase1.run_topic_discovery --step review
./.venv/bin/python -m phase1.run_topic_discovery --step sensitivity
./.venv/bin/python -m phase1.run_topic_discovery --step document
```

人工标注完成后：

```bash
./.venv/bin/python -m phase1.run_topic_discovery --step agreement
./.venv/bin/python -m phase1.run_topic_discovery --step backfill
```

## 1. 选择哪个 HDBSCAN 结果

选择标准不是“topic 越少越好”，而是能否支持人工开放归纳：

- topic 数量要可审阅，最好几十到一百左右；
- 最大簇不能吞掉太多语义，否则人工归并会被一个泛化桶主导；
- topic 的 top terms 与代表评论应能给 coder 足够语义线索；
- outlier 可以高，但必须明确主分析使用 raw topics 还是 outlier-reassigned topics。

这里建议主分析使用 raw HDBSCAN topics 做人工归并，`topic_assigned` 只作为辅助检查。原因是 raw `-1` 表示模型认为边界不足的评论；把 outliers 强行分配进 topic 会提高覆盖率，但会降低主题边界的解释纯度。

In [2]:
def parse_terms(x, n=10):
    if pd.isna(x):
        return []
    s = str(x)
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)):
            return [str(t).strip() for t in obj if str(t).strip()][:n]
    except Exception:
        pass
    return [t.strip() for t in s.split(",") if t.strip()][:n]


def load_hdbscan(mcs: int):
    d = RUN_DIR / f"bert_hdbscan_mcs{mcs}"
    topics = pd.read_csv(d / "topics.csv")
    docs = pd.read_csv(d / "doc_topics.csv")
    samples = pd.read_csv(d / "samples.csv")
    return d, topics, docs, samples


def candidate_summary(mcs: int) -> dict:
    _, topics, docs, _ = load_hdbscan(mcs)
    valid = topics[topics["Topic"] >= 0].copy()
    counts = valid["Count"].to_numpy(dtype=float)
    total = len(docs)
    raw_out = float((docs["topic_raw"] == -1).mean()) if "topic_raw" in docs else np.nan
    assigned_out = float((docs["topic_assigned"] == -1).mean()) if "topic_assigned" in docs else np.nan
    return {
        "mcs": mcs,
        "n_docs": total,
        "n_topics": int(len(valid)),
        "raw_outlier_rate": raw_out,
        "assigned_outlier_rate": assigned_out,
        "valid_docs": int(counts.sum()),
        "largest_topic": int(counts.max()),
        "largest_share_valid": float(counts.max() / counts.sum()),
        "largest_share_all": float(counts.max() / total),
        "median_topic_size": float(np.median(counts)),
        "p25_topic_size": float(np.percentile(counts, 25)),
        "p75_topic_size": float(np.percentile(counts, 75)),
        "min_topic_size": int(counts.min()),
        "n_topics_ge_80": int((counts >= 80).sum()),
        "n_topics_ge_100": int((counts >= 100).sum()),
    }

summary = pd.DataFrame([candidate_summary(mcs) for mcs in CANDIDATE_MCS])
display(summary.round(4))
summary.to_csv(OUT_DIR / "hdbscan_candidate_summary.csv", index=False)

,mcs,n_docs,n_topics,raw_outlier_rate,assigned_outlier_rate,valid_docs,largest_topic,largest_share_valid,largest_share_all,median_topic_size,p25_topic_size,p75_topic_size,min_topic_size,n_topics_ge_80,n_topics_ge_100
0,50,26811,88,0.4442,0.0094,14902,1129,0.0758,0.0421,107.0,65.75,188.5,50,58,50
1,80,26811,37,0.3751,0.0084,16753,6601,0.3940,0.2462,191.0,125.00,333.0,82,37,32


In [3]:
def topic_preview(mcs: int, n_topics: int = 30) -> pd.DataFrame:
    _, topics, _, _ = load_hdbscan(mcs)
    valid = topics[topics["Topic"] >= 0].copy().head(n_topics)
    valid["top_terms"] = valid["Representation"].map(lambda x: ", ".join(parse_terms(x, n=10)))
    cols = ["Topic", "Count", "Name", "top_terms", "Representative_Docs"]
    return valid[cols]

for mcs in CANDIDATE_MCS:
    print(f"\n=== HDBSCAN mcs={mcs}: first 30 topics ===")
    display(topic_preview(mcs, 30))


=== HDBSCAN mcs=50: first 30 topics ===


,Topic,Count,Name,top_terms,Representative_Docs
1,0,1129,0_遥控_遥控器_遥控车_遙控,"遥控, 遥控器, 遥控车, 遙控, 监控, 失控, 操控, 有人, 跟不上, 跟着","['[微笑R]这不都是人遥控的么', '哈哈哈，遥控de', '有自主跑的，也有遥控的。']"
2,1,646,1_机器人马拉松_机器人_马拉松_人形机器人,"机器人马拉松, 机器人, 马拉松, 人形机器人, 跑马, 机器, 跑步, 赛事, 奔跑, 比赛","['机器人跑马拉松[偷笑R]', '以后马拉松就是机器人比赛[偷笑R]', '有的。\n国外..."
3,2,560,2_机器人_小机器人_机器人界_機器,"机器人, 小机器人, 机器人界, 機器, 机器, 人机, 真好玩, 机械, 好笑, 有个","['机器人添大乱', '机器人：力大砖飞[坏笑R]', '机器人不好玩！']"
4,3,499,3_程序员_跟着_跟不上_跑不动,"程序员, 跟着, 跟不上, 跑不动, 跑不过, 跑啦, 编程, 程序, 追着, run","['程序员咋不跟着跑', '程序员也要跟着跑步，可能是救程序员的', '程序员说写的时候没有..."
5,4,498,4_战场_打仗_作战_战士,"战场, 打仗, 作战, 战士, 战时, 战争, 牺牲, 背着, 军用, 武器","['意义就是以后战场不需要人了[doge]', '战场上背炸弹', '对啊，让他跑这么快的意..."
6,5,494,5_人形机器人_人形_机器人_人型,"人形机器人, 人形, 机器人, 人型, 人体, 外形, 人类, 形状, 机器, 形态","['就是因为人形才叫机器人啊傻不傻', '人形设计出来是为了代替人类的，整个世界的各种工具都..."
7,6,435,6_摔倒_摔了一跤_跌倒_摔到,"摔倒, 摔了一跤, 跌倒, 摔到, 摔得, 摔下去, 摔成, 摔个, 摔碎, 绊倒","['总感觉下一秒就要摔倒了，一直到最后也没摔倒', '真不是故意摔倒的？', '摔倒了的那个..."
8,7,418,7_机器人_机器_人类_人工,"机器人, 机器, 人类, 人工, 服务, 人来, 伺候, 服务业, 有个, 工作","['以后人类要服务机器人了！', '所以是机器人服务人类还是人类服务机器人[微笑R]', '..."
9,8,390,8_去年_今年_昨年_前年,"去年, 今年, 昨年, 前年, 第二年, 跑的快, 飞速发展, 好快, 进步, 之前","['去年他还需要人扶，今年都能跑马拉松了[赞R]', '今年的比去年的快多了。', '哈哈哈..."
10,9,370,9_轮子_更快_更稳_轮胎,"轮子, 更快, 更稳, 轮胎, 坐轮椅, 更好, 上不去, 轮式, 轮椅, 不比","['轮子不更快', '轮子不是更快吗？为啥要手，脚的', '那4个轮子不是更快[doge]']"



=== HDBSCAN mcs=80: first 30 topics ===


,Topic,Count,Name,top_terms,Representative_Docs
1,0,6601,0_机器人_人形机器人_小机器人_机器人马拉松,"机器人, 人形机器人, 小机器人, 机器人马拉松, 机器狗, 机器, 机械, 人形, 人类, 人型","['机器人大逃亡', '遛人还是溜机器人？', '机器人，人啊']"
2,1,1657,1_降温_降降温_加冰_冰块,"降温, 降降温, 加冰, 冰块, 发热, 冷却, 高温, 温度, 过热, 中暑","['倒冰块降温吧', '冰块降温吗？[偷笑R]', '降温散热的']"
3,2,1099,2_遥控_遥控器_遥控车_遙控,"遥控, 遥控器, 遥控车, 遙控, 监控, 操控, 玩意, 有人, 摇控, 控制","['还有遥控的，假装能自己动', '所以还是遥控的[捂脸R]还得人陪着跑', '怎么都是遥控..."
4,3,536,3_摔倒_跌倒_摔到_摔了一跤,"摔倒, 跌倒, 摔到, 摔了一跤, 摔得, 摔成, 摔碎, 摔死, 撞到, 站不稳","['感觉随时想摔倒。', '总感觉下一秒就要摔倒了，一直到最后也没摔倒', '真不是故意摔倒..."
5,4,501,4_战场_打仗_作战_战士,"战场, 打仗, 作战, 战士, 牺牲, 没用, 战争, 用来, 意义, 要是","['意义就是以后战场不需要人了[doge]', '对啊，让他跑这么快的意义是？上战场吗', ..."
6,5,476,5_程序员_跟着_编程_跟不上,"程序员, 跟着, 编程, 跟不上, 跑不过, 程序, 跑不动, 码农, 代码, run","['程序员也要跟着跑步，可能是救程序员的', '程序员说写的时候没有这个变量', '[捂脸R..."
7,6,386,6_跑得慢_跑不动_跑动_跑步,"跑得慢, 跑不动, 跑动, 跑步, 姿势, 跑法, 奔跑, 步伐, 跑道, 动作","['这个跑步姿势有点丑[石化R][石化R][石化R]', '哈哈哈哈，章跑步就这个姿势', ..."
8,7,383,7_好笑_搞笑_好好笑_真好玩,"好笑, 搞笑, 好好笑, 真好玩, 有趣, 好玩, 逗笑, 有意思, 笑点, 太逗","['真的有点好笑了', '多拍点，真的好搞笑[笑哭R][笑哭R]', '这个真的太搞笑了[笑..."
9,8,343,8_比赛_参赛_这比_大赛,"比赛, 参赛, 这比, 大赛, 观赛, 赛制, 参赛选手, 赛前, 有意思, 有意义","['比不比赛了还[笑哭R]哈哈哈哈光饭撒了', '这是比谁的尸块多比赛[笑哭R]', '哈哈..."
10,9,333,9_去年_今年_昨年_前年,"去年, 今年, 昨年, 前年, 第二年, 一年, 每年, 之前, 跑的快, 明年","['今年的比去年的快多了。', '比去年可进步多了', '哈哈哈哈哈哈哈去年也是这么说的，去..."


## 2. 主候选：`mcs=50`

`mcs=50` 的优点是 topic 仍然细：遥控/自主、战场化想象、人形必要性、摔倒、服务与伺候、技术进步、电池续航、降温散热、养老、补贴质疑、恐怖谷等都能分开出现。这个粒度适合让 coder 做归并，而不是让算法提前决定大类。

审阅表基于 `doc_topics.csv` 的 **raw topic** 分配，并从 `shared_analyzable_corpus.csv` 补回 `like_count`、`comment_level` 等字段。每个 topic 提供三类样本与覆盖诊断，避免只被高赞或单帖带偏。

## 3. Coder 归并规则草案

给 coder 的任务不是判断评论立场，而是给每个 topic 归并一个更高层的话语主题。建议每个 topic 只给一个主 domain；如果 topic 混杂，标记为 `Mixed/Unclear`，并在 notes 写明混杂来源。

建议初始 domain 保持开放，可在第一轮后合并：

- `technical capability`：自主、遥控、算法、跑姿、稳定性、续航、散热等；
- `human support / maintenance`：程序员、跟跑、抢救、担架、多人伺候、补给；
- `affect / anthropomorphism`：可爱、心疼、像小孩、拟人化身体与情绪；
- `humor / spectacle`：搞笑、解说、春晚、小品、梗文化；
- `social future / progress`：科技进步、国家发展、未来应用、几年后想象；
- `risk / militarization / governance`：战场、终结者、控制、危险、补贴质疑；
- `human identity / labor`：替代人类、人还有何意义、养老、服务关系倒置；
- `brand / event-specific`：品牌、赛事组织、地点、特定参赛者；
- `mixed / unclear`：top terms 和样本无法形成单一可解释主题。

## 2b. 审阅表：高赞 + 跨帖 + 随机

每个 topic 的审阅材料包括：

- `top_terms` / `representative_docs`：BERTopic 关键词与代表文本；
- `top_like_examples`：topic 内最高赞评论，反映社会可见度；
- `post_diverse_examples`：每个帖子最多 1 条（优先高赞），看跨帖重复语义；
- `random_examples`：固定随机种子抽样，降低高赞与单帖偏差；
- `n_posts_raw` / `top_post_share_raw`：该 topic 覆盖多少帖、是否被少数爆款帖支配；
- `level1_share_raw` / `level2_share_raw`：一级评论 vs 二级回复比例。

人工归并时，建议以 `top_terms + representative_docs + post_diverse_examples + random_examples` 共同判断主题边界；`top_like_examples` 只作可见度参考。

In [ ]:
def format_examples(df: pd.DataFrame, n: int, *, include_meta: bool = True) -> str:
    """Format comments as a compact block for coder review."""
    if df.empty:
        return ""
    rows = []
    for _, r in df.head(n).iterrows():
        text = str(r.get("content", "")).replace("\n", " ").strip()
        if include_meta:
            meta = (
                f"post={r.get('帖子id', '')}; "
                f"level={r.get('comment_level', '')}; "
                f"likes={r.get('like_count', '')}; "
                f"status={r.get('robot_status', '')}; role={r.get('human_role', '')}"
            )
            rows.append(f"[{meta}] {text}")
        else:
            rows.append(text)
    return "\n---\n".join(rows)


def docs_with_full_metadata(docs: pd.DataFrame) -> pd.DataFrame:
    """`doc_topics.csv` lacks like_count/comment_level; recover them from shared corpus."""
    shared = pd.read_csv(SHARED_CORPUS_PATH)
    meta_cols = [
        "comment_id",
        "comment_level",
        "like_count",
        "reply_count",
        "char_len",
        "location",
        "robot_status_group",
    ]
    keep = [c for c in meta_cols if c in shared.columns]
    return docs.merge(shared[keep], on="comment_id", how="left", suffixes=("", "_shared"))


def build_enhanced_topic_review_table(
    mcs: int,
    *,
    top_like_n: int = 5,
    post_diverse_n: int = 8,
    random_n: int = 5,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    _, topics, docs, _ = load_hdbscan(mcs)
    docs = docs_with_full_metadata(docs)
    valid_topics = topics[topics["Topic"] >= 0].copy()
    valid_topics["top_terms"] = valid_topics["Representation"].map(lambda x: ", ".join(parse_terms(x, n=12)))

    raw_docs = docs[docs["topic_raw"] >= 0].copy()
    raw_docs["topic_raw"] = raw_docs["topic_raw"].astype(int)
    raw_docs["like_count"] = pd.to_numeric(raw_docs["like_count"], errors="coerce").fillna(0)
    raw_docs["comment_level"] = pd.to_numeric(raw_docs["comment_level"], errors="coerce")

    rows = []
    post_rows = []
    rng = np.random.default_rng(random_state)

    for _, row in valid_topics.iterrows():
        tid = int(row["Topic"])
        sub = raw_docs[raw_docs["topic_raw"] == tid].copy()
        sub_like = sub.sort_values(["like_count", "comment_id"], ascending=[False, True])
        top_like = sub_like.head(top_like_n)

        # Cross-post examples: best liked comment per post, then keep top posts.
        post_diverse = (
            sub_like.drop_duplicates("帖子id", keep="first")
            .sort_values(["like_count", "comment_id"], ascending=[False, True])
            .head(post_diverse_n)
        )

        if len(sub) <= random_n:
            random_ex = sub.sample(frac=1.0, random_state=random_state)
        else:
            # Seed by topic id for stable but topic-specific random examples.
            random_ex = sub.sample(n=random_n, random_state=int(rng.integers(0, 1_000_000) + tid))

        post_counts = sub.groupby("帖子id").size().sort_values(ascending=False)
        n_posts = int(post_counts.size)
        top_post_share = float(post_counts.iloc[0] / len(sub)) if len(sub) else np.nan
        level_counts = sub["comment_level"].value_counts(normalize=True)
        level1_share = float(level_counts.get(1, 0.0))
        level2_share = float(level_counts.get(2, 0.0))

        for pid, cnt in post_counts.head(5).items():
            post_rows.append({"topic_id": tid, "帖子id": pid, "raw_count": int(cnt), "share_raw": float(cnt / len(sub))})

        rows.append(
            {
                "topic_id": tid,
                "raw_count": int(len(sub)),
                "assigned_count": int((docs["topic_assigned"] == tid).sum()) if "topic_assigned" in docs else np.nan,
                "n_posts_raw": n_posts,
                "top_post_share_raw": top_post_share,
                "level1_share_raw": level1_share,
                "level2_share_raw": level2_share,
                "top_terms": row["top_terms"],
                "representative_docs": row.get("Representative_Docs", ""),
                "top_like_examples": format_examples(top_like, top_like_n),
                "post_diverse_examples": format_examples(post_diverse, post_diverse_n),
                "random_examples": format_examples(random_ex, random_n),
                "coder1_domain": "",
                "coder1_label": "",
                "coder1_notes": "",
                "coder2_domain": "",
                "coder2_label": "",
                "coder2_notes": "",
                "adjudicated_domain": "",
                "adjudicated_label": "",
                "adjudication_notes": "",
            }
        )

    review_df = pd.DataFrame(rows).sort_values("raw_count", ascending=False).reset_index(drop=True)
    post_diag_df = pd.DataFrame(post_rows).sort_values(["topic_id", "raw_count"], ascending=[True, False])
    return review_df, post_diag_df

enhanced_review, topic_post_diag = build_enhanced_topic_review_table(PRIMARY_MCS)

enhanced_path = OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_topic_review_sheet_enhanced.csv"
post_diag_path = OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_topic_post_diagnostics.csv"
enhanced_review.to_csv(enhanced_path, index=False)
topic_post_diag.to_csv(post_diag_path, index=False)

print(f"saved: {enhanced_path}")
print(f"saved: {post_diag_path}")
display(enhanced_review.head(10))

In [ ]:
# Enhanced blank coder sheets: richer evidence, one independent file per coder.
enhanced_base_cols = [
    "topic_id",
    "raw_count",
    "assigned_count",
    "n_posts_raw",
    "top_post_share_raw",
    "level1_share_raw",
    "level2_share_raw",
    "top_terms",
    "representative_docs",
    "top_like_examples",
    "post_diverse_examples",
    "random_examples",
]

for coder in [1, 2]:
    sheet = enhanced_review[enhanced_base_cols].copy()
    sheet["domain"] = ""
    sheet["label"] = ""
    sheet["notes"] = ""
    path = OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_coder{coder}_sheet_enhanced.csv"
    sheet.to_csv(path, index=False)
    print("saved:", path)

In [ ]:
# After coding: place completed coder sheets in OUT_DIR and run this cell.
# Expected columns in each sheet: topic_id, domain, label, notes
from sklearn.metrics import cohen_kappa_score

coder1_done = OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_coder1_sheet_enhanced_done.csv"
coder2_done = OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_coder2_sheet_enhanced_done.csv"

if coder1_done.exists() and coder2_done.exists():
    c1 = pd.read_csv(coder1_done)
    c2 = pd.read_csv(coder2_done)
    merged = c1[["topic_id", "domain", "label", "notes"]].rename(
        columns={"domain": "coder1_domain", "label": "coder1_label", "notes": "coder1_notes"}
    ).merge(
        c2[["topic_id", "domain", "label", "notes"]].rename(
            columns={"domain": "coder2_domain", "label": "coder2_label", "notes": "coder2_notes"}
        ),
        on="topic_id",
        how="inner",
    )
    domain_kappa = cohen_kappa_score(merged["coder1_domain"].fillna(""), merged["coder2_domain"].fillna(""))
    label_kappa = cohen_kappa_score(merged["coder1_label"].fillna(""), merged["coder2_label"].fillna(""))
    print(f"Domain Cohen's kappa: {domain_kappa:.3f}")
    print(f"Label Cohen's kappa:  {label_kappa:.3f}")
    merged["domain_agree"] = merged["coder1_domain"] == merged["coder2_domain"]
    merged["label_agree"] = merged["coder1_label"] == merged["coder2_label"]
    display(merged)
    merged.to_csv(OUT_DIR / f"hdbscan_mcs{PRIMARY_MCS}_coder_agreement.csv", index=False)
else:
    print("Coder done files not found yet:")
    print(coder1_done)
    print(coder2_done)

## 4. 报告写法建议

如果后续采用 `mcs=50`，方法段可以写成：

> We used BERTopic with BGE Chinese sentence embeddings and HDBSCAN to generate fine-grained, data-driven topics. We selected the `min_cluster_size=50` solution for manual review because it yielded a manageable number of coherent topics while avoiding a dominant residual cluster. Two coders independently reviewed each topic using its top terms and representative comments, assigned a higher-order discourse domain and an inductive label, and resolved disagreements through adjudication. Inter-coder agreement was assessed using Cohen's kappa.

需要避免的写法：

- 不要说 HDBSCAN 自动发现了最终 discourse domains；最终 domain 是人工归并结果。
- 不要把 outlier-reassigned topics 当作唯一结果；报告时说明 raw outlier rate 和是否使用 reassignment。
- 不要把既有理论 domain 写成模型训练标签；它们最多是 adjudication 阶段的解释资源。